# Trabajo en clase — Q-Learning con FrozenLake

En CheeseWorld construimos el algoritmo desde cero. Ahora utilizaremos el mismo procedimiento en un ambiente estándar de **Gymnasium**.

## Objetivo

Durante la clase debes relacionar cada parte del código con los conceptos:

- estado \(s\);
- acción \(a\);
- recompensa \(r\);
- Q-table;
- exploración y explotación;
- TD target;
- TD error;
- política greedy.

Este notebook tiene **espacios para discutir y escribir conclusiones durante la clase**.


## 1. Imports y funciones de Q-Learning


In [1]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt


def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))


def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)


def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]

            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards


def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

### 💬 Antes de ejecutar

En CheeseWorld teníamos explícitamente una clase `Environment` y una clase `QLearningAgent`.

**Pregunta:** en este notebook, ¿qué papel cumple Gymnasium y dónde quedó representado el agente?

**Notas:**

Gymnasium cumple el papel del ambiente, o sea lo que antes era la clase `Environment` de CheeseWorld. Con `env.reset()` y `env.step(action)` nos da el estado, la recompensa y si el episodio terminó, y nosotros no tenemos que programar las reglas del lago.

El agente ya no es una clase, quedó repartido en funciones: la Q table que crea `initialize_q_table`, la forma de escoger acciones en `epsilon_greedy_policy` y `greedy_policy`, y el aprendizaje en `train_q_learning`, que es donde se hace la actualización con el TD target.

## 2. Crear FrozenLake


In [2]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

state, info = env.reset()

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

Estado inicial: 0
Número de estados: 16
Número de acciones: 4


En FrozenLake las acciones son:

| Acción | Código |
|---|---:|
| Left | 0 |
| Down | 1 |
| Right | 2 |
| Up | 3 |

Primero trabajaremos con `is_slippery=False`, es decir, con transiciones determinísticas.


### 💬 Actividad 1 — La Q-table

Antes de crearla:

1. ¿Cuántas filas debe tener la Q-table?
2. ¿Cuántas columnas?
3. ¿Qué representa una celda \(Q[s,a]\)?

**Respuesta / discusión:**

1. Debe tener 16 filas, una por cada estado, porque el mapa es de 4x4 casillas.
2. Debe tener 4 columnas, una por cada acción (Left, Down, Right, Up).
3. Q[s,a] es lo que el agente estima que va a ganar en total (con descuento) si está en el estado s, hace la acción a y después sigue actuando de la mejor forma.

In [3]:
state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
Q

Q-table shape: (16, 4)


array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

### 💬 Actividad 2 — Inicio del aprendizaje

Todos los valores son cero.

$$
Q(s,a)=0
$$

¿Esto significa que todas las acciones son malas, o que el agente todavía no sabe nada?

¿Qué ocurre si varias acciones tienen exactamente el mismo valor máximo?

**Notas:**

Que todo esté en cero no quiere decir que las acciones sean malas, quiere decir que el agente todavía no sabe nada porque no ha tenido ninguna experiencia. Es solo el valor inicial.

Si varias acciones tienen el mismo valor máximo, la función `greedy_policy` desempata al azar con `np.random.choice`. Por eso al principio el agente greedy se comporta casi como uno aleatorio. Si siempre se escogiera la primera acción (Left) el agente se quedaría pegado contra la pared.

## 3. Ejecutar una transición


In [4]:
state, _ = env.reset()

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)

state      = 0
action     = 0
reward     = 0
next_state = 0
done       = False


### 💬 Actividad 3 — Identificar la experiencia

Escribe la experiencia anterior como:

$$
(s,a,r,s')
$$

**Experiencia:**

$$
(0,\ 0,\ 0,\ 0)
$$

En la ejecución el agente estaba en el estado 0, escogió la acción 0 (Left), recibió recompensa 0 y como se chocó con el borde quedó otra vez en el estado 0.

¿De cuál de esos cuatro elementos **no disponíamos directamente** en Value Iteration cuando hablábamos de experiencia real?

**Respuesta:** el elemento que no teníamos directamente es s', el estado siguiente que sale de la interacción real. En Value Iteration conocíamos T(s,a,s') y R(s), entonces podíamos calcular el promedio sobre todos los s' posibles sin movernos. Aquí no hay modelo, el ambiente nos devuelve un solo s' (y su recompensa) y con esa muestra aprendemos.

## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [5]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


Snapshots guardados:
Q_initial : 0 episodios
Q_early   : 50 episodios
Q_trained : 10000 episodios


### Momento 1 — Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [6]:
frames_random, reward_random = play_episode(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
frames_to_video(frames_random, interval=700)


Recompensa total: 0.0


### Momento 2 — Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [7]:
frames_early, reward_early = play_episode(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
frames_to_video(frames_early, interval=700)


Recompensa total: 0.0


### Momento 3 — Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [8]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
frames_to_video(frames_trained, interval=700)


Recompensa total: 1.0


### 💬 Actividad — ¿Qué cambió?

Compara las tres ejecuciones. El ambiente, los estados y las acciones son los mismos. **¿Qué cambió internamente en el agente para que su comportamiento mejore?**

Observa `Q_initial`, `Q_early` y `Q_trained` y relaciona sus valores con las acciones que viste ejecutar.


**Respuesta:** lo único que cambió fue la Q table. En `Q_initial` todo es cero. Algo que me llamó la atención es que `Q_early` también quedó toda en cero: en los primeros 50 episodios el agente nunca llegó a la meta y como la única recompensa es el 1 de la meta, no hubo nada que propagar. Por eso en el momento 2 el agente se comporta igual que el aleatorio y la recompensa fue 0.

En `Q_trained` ya aparecen valores que crecen a medida que el estado está más cerca de la meta (0.774 en el inicio, 0.95 al lado de la meta y 1 en la casilla 14 justo antes). Son potencias de 0.95, que es el gamma. Al escoger siempre la acción con mayor Q, el agente sigue ese camino de valores crecientes y llega a la meta, por eso la recompensa fue 1.

In [9]:
flechas = np.array(["<", "v", ">", "^"])

def mostrar_politica(Q):
    pol = flechas[np.argmax(Q, axis=1)]
    pol[np.max(Q, axis=1) == 0] = "."
    print(pol.reshape(4, 4))

print("Valor maximo de Q en cada estado (Q_initial):")
print(np.round(Q_initial.max(axis=1).reshape(4, 4), 3))
print("\nValor maximo de Q en cada estado (Q_early):")
print(np.round(Q_early.max(axis=1).reshape(4, 4), 3))
print("\nValor maximo de Q en cada estado (Q_trained):")
print(np.round(Q_trained.max(axis=1).reshape(4, 4), 3))

print("\nPolitica greedy Q_early ('.' = todo en cero):")
mostrar_politica(Q_early)
print("\nPolitica greedy Q_trained:")
mostrar_politica(Q_trained)

Valor maximo de Q en cada estado (Q_initial):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

Valor maximo de Q en cada estado (Q_early):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

Valor maximo de Q en cada estado (Q_trained):
[[0.774 0.815 0.857 0.815]
 [0.815 0.    0.902 0.   ]
 [0.857 0.902 0.95  0.   ]
 [0.    0.95  1.    0.   ]]

Politica greedy Q_early ('.' = todo en cero):
[['.' '.' '.' '.']
 ['.' '.' '.' '.']
 ['.' '.' '.' '.']
 ['.' '.' '.' '.']]

Politica greedy Q_trained:
[['v' '>' 'v' '<']
 ['v' '.' 'v' '.']
 ['>' 'v' 'v' '.']
 ['.' '>' '>' '.']]


### 💬 Actividad 4 — Leer una fila de Q

Selecciona un estado $s$ y observa:

$$
Q(s,0), Q(s,1), Q(s,2), Q(s,3)
$$

**Estado seleccionado:** 0 (la casilla de inicio)

**Valores Q:**

Left: 0.7351, Down: 0.7738, Right: 0.7738, Up: 0.7351

¿Cuál acción seleccionaría:

$$
\arg\max_a Q(s,a)
$$

?

**Interpretación:**

> Hay un empate entre Down y Right con 0.7738. `np.argmax` devuelve Down porque es la primera, pero la política greedy del notebook desempata al azar, así que puede escoger cualquiera de las dos. Tiene sentido porque desde la esquina hay dos caminos igual de cortos (6 pasos) hacia la meta, y 0.95 elevado a 5 da 0.7738. Left y Up valen menos porque chocan con la pared y se pierde un paso.

In [10]:
s = 0
nombres = ["Left", "Down", "Right", "Up"]
print("Estado:", s)
for a in range(4):
    print(f"{nombres[a]:5s}: {Q_trained[s, a]:.4f}")
print("argmax:", nombres[np.argmax(Q_trained[s])])

Estado: 0
Left : 0.7351
Down : 0.7738
Right: 0.7738
Up   : 0.7351
argmax: Down


## 5. Evaluar la política aprendida


In [11]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


Mean reward: 1.000
Std reward : 0.000


### 💬 Actividad 5 — Exploration vs. exploitation

Durante entrenamiento usamos $\epsilon$-greedy.

Durante evaluación usamos:

$$
a=\arg\max_aQ(s,a)
$$

¿Por qué **no exploramos** durante la evaluación?

**Conclusión:**

> Porque en la evaluación queremos medir qué tan buena es la política que el agente ya aprendió. Si seguimos explorando, algunas acciones serían al azar y la recompensa bajaría por culpa de la exploración, no por lo que aprendió. La exploración solo sirve mientras se aprende. Con la política greedy el agente determinístico llegó a la meta en los 100 episodios (media 1.000 y desviación 0).

## 6. Experimento: FrozenLake estocástico


In [12]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

Mean reward: 0.328
Std reward : 0.469


### 💬 Actividad 6 — Determinístico vs. estocástico

Compara:

- `is_slippery=False`
- `is_slippery=True`

¿Qué cambia en el **ambiente**?

¿Qué cambia en la **ecuación de Q-Learning**?

**Discusión:**

**Ambiente:** con `is_slippery=True` el lago resbala y la acción que se escoge no siempre es la que pasa, el agente se puede ir para un lado. Entonces el mismo (s,a) puede llevar a distintos s'. Eso se ve en el resultado: el determinístico tuvo media 1.000 y el resbaloso solo 0.328 con desviación 0.469, o sea que gana más o menos 1 de cada 3 episodios aunque use la mejor política que encontró. También necesitó más episodios (20000) y un decay más lento.

**Algoritmo:** la ecuación de Q learning es exactamente la misma. No hay que cambiar nada porque nunca usamos las probabilidades de transición, simplemente las muestras que llegan ya traen la aleatoriedad y Q termina aprendiendo un promedio.

# Cierre de clase

Completa antes de terminar:

**1. ¿Qué almacena $Q(s,a)$?**

> La estimación de la recompensa total descontada que se obtiene si estando en s se hace la acción a y después se sigue la mejor política.

**2. ¿De dónde sale $\max_{a'}Q(s',a')$?**

> Sale de la misma Q table, en la fila del estado siguiente s' que nos devolvió el ambiente. Es lo mejor que el agente cree que puede lograr desde ahí, y se usa para armar el TD target junto con la recompensa.

**3. ¿Por qué necesitamos $\epsilon$-greedy?**

> Porque si el agente solo explota lo que sabe, nunca prueba acciones nuevas y se puede quedar con una política mala. Con epsilon greedy al principio explora mucho (epsilon cerca de 1) y descubre la meta, y después va bajando epsilon para aprovechar lo aprendido.

**4. ¿Por qué Q-Learning es model-free?**

> Porque no necesita conocer T(s,a,s') ni la función de recompensa. Aprende directamente de las experiencias (s, a, r, s') que va obteniendo al interactuar con el ambiente, a diferencia de Value Iteration, donde teníamos el modelo completo.